In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
import math
import faultdiagnosistoolbox as fdt
import time
import os
import sys
import scipy.io as sio
import sympy as sym
from scipy.io import loadmat
from utils.data_utils import create_sequences, load_and_normalize_data, sorting_key
import pickle



In [ ]:
plt.rcdefaults()
plt.rcParams["text.latex.preamble"] = r""
plt.rcParams["text.usetex"] = True
plt.rcParams["lines.linewidth"] = 2
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = 8.
plt.rcParams['legend.fontsize'] = 8.
# mpl.rcParams.update(mpl.rcParamsDefault)


%matplotlib qt

In [ ]:
# # You need to install https://faultdiagnosistoolbox.github.io in your python installation
# # pip install faultdiagnosistoolbox
# # should work

# # %% Import model
# import matplotlib.pyplot as plt
# from engine_model import LiU_ICE_model

# # %% Basic fault diagnosis toolbox evaluations
# # Model summary
# LiU_ICE_model.Lint()

# # Plot structural model
# plt.subplots()
# LiU_ICE_model.PlotModel()

# # % Plot Dulmage-Mendelsohn decomposition with equivalence classes and faults
# plt.subplots()
# LiU_ICE_model.PlotDM(fault=True, eqclass=True)

# # %%
# plt.show()

In [ ]:
def load_results(r, M, n_epochs_MSE, n_epochs_NLL, initial_seq_length, max_seq_length, seq_length_step, results_dir="save_models"):
    """Load results from the specified directory."""
    # Construct the file path
    file_name = f"results_{r}_M{M}_epochs{n_epochs_MSE}.pkl"
    results_path = os.path.join(results_dir, f"ensemble_{r}_M{M}_epochs{n_epochs_MSE}_{n_epochs_NLL}_seq{initial_seq_length}_{max_seq_length}_{seq_length_step}", "results", file_name)
    
    # Check if the file exists
    if not os.path.exists(results_path):
        raise FileNotFoundError(f"Results file not found: {results_path}")
    
    # Load the results
    with open(results_path, "rb") as f:
        results = pickle.load(f)
    
    return results


In [ ]:
from engine_model import LiU_ICE_model
diag_model = LiU_ICE_model
msos = diag_model.MSO()
mtes = diag_model.MTES()
li = [m for m in msos if diag_model.IsLowIndex(m)]
ts = diag_model.TestSelection(li)
ts_msos = [li[ts_i] for ts_i in ts]
FSM = diag_model.FSM(ts_msos, plot=False)

In [ ]:
msos_list = [msos[590], msos[787], msos[1634], msos[976], msos[980], msos[1667], msos[1673], msos[2096]]
FSM = diag_model.FSM(msos_list, plot=True)
print(FSM)

In [ ]:
msos_list = [msos[590], msos[787], msos[1634], msos[976], msos[980], msos[1667], msos[1673]]
FSM = diag_model.FSM(msos_list, plot=False)

ts = [590, 787, 1634, 976, 980, 1667, 1673]
re = [84, 86, 87, 86, 84, 87, 84]
for msoIdx, redIdx in zip(ts, re):
    mso = msos[msoIdx]
    indx = msos[msoIdx].index(redIdx)
    red = mso[indx]
    m0 = [e for e in mso if e != red]
    Gamma = diag_model.Matching(m0)
    print(f"MSO {msoIdx} with redundant equation {red}, causality: {Gamma.matchType}")
FSM = diag_model.FSM([msos[ti] for ti in ts])

In [ ]:
textwidth = 88 / 25.4  # matplotlib deals with inches
golden_ratio = (1 + np.sqrt(5)) / 2 
figsize = (0.95 * textwidth, 0.95 * textwidth / golden_ratio )
fig, (ax) = plt.subplots(1, 1, figsize=figsize, clear=True, layout="compressed")
FSM = diag_model.FSM(msos_list, plot=False)
diag_model.FSM(msos_list, plot=True)
# Customize y-axis labels
ax.set_yticks([0.01, 1, 2])  # Specify positions
ax.set_yticklabels(['r0', 'r1', 'r2'])  # Specify new labels

In [ ]:
# Iterate through all .mat files in the directory
directory = 'Engine_data/trainingdata/'
combined_data = load_and_normalize_data(directory)
signal_name = [
    "time",                     # Time (no short name provided)
    "ypic",                     # Intercooler pressure
    "yTic",                     # Intercooler temperature
    "ypim",                     # Intake manifold pressure
    "ywaf",                     # Mass flow through the air filter
    "yomega",                   # Engine speed
    "yxpos",                    # Throttle actuator position
    "uwg",                      # Requested wastegate actuator position
    "umf",                      # Requested injected fuel mass
    "ypamb",                    # Ambient pressure
    "yTamb"                     # Ambient temperature
]
data_name = sorting_key(combined_data)

setting = {"r": "r0", "M": 1, "epochs_MSE": 20, "epochs_NLL": 20, "initial_seq_length": 100, "max_seq_length": 800, "seq_length_step": 100}


r = setting["r"]
M = setting["M"]
n_epochs_MSE = setting["epochs_MSE"]
n_epochs_NLL = setting["epochs_NLL"]
initial_seq_length = setting["initial_seq_length"]
max_seq_length = setting["max_seq_length"]
seq_length_step = setting["seq_length_step"]


# Load results
results = load_results(r, M, n_epochs_MSE, n_epochs_NLL, initial_seq_length, max_seq_length, seq_length_step)
all_mu_star = results['all_mu_star']
all_sigma_star = results['all_sigma_star']
all_y_test = results['all_y_test']
all_U_aleatoric = results['all_U_aleatoric']
all_U_epistemic = results['all_U_epistemic']
all_r = results['all_r']

# Function to calculate maximum absolute value ignoring the top 2% outliers
def max_abs_exclude_outliers(data, top_percentile=2):
    percentile_cutoff = np.percentile(np.abs(data), 100 - top_percentile)
    filtered_data = data[np.abs(data) <= percentile_cutoff]
    return np.max(np.abs(filtered_data))

# Apply the function to each dataset
r_N = max_abs_exclude_outliers(all_r[-1])
U_alea_N = max_abs_exclude_outliers(all_U_aleatoric[-1])
U_epi_N = max_abs_exclude_outliers(all_U_epistemic[-1])
print(f'r_N: {r_N}, U_alea_N: {U_alea_N}, U_epi_N: {U_epi_N}')

all_r = [[x / r_N for x in sub_array] for sub_array in all_r]
Thresholds = [[np.sqrt(x) / r_N for x in sub_array] for sub_array in all_U_aleatoric]
all_U_aleatoric = [[x / U_alea_N for x in sub_array] for sub_array in all_U_aleatoric]
all_U_epistemic = [[x / U_epi_N for x in sub_array] for sub_array in all_U_epistemic]


In [ ]:
ignore = 40
# Define grid dimensions
FSM_with_NF = np.array([
    [1, 1, 1, 0, 0, 1, 1, 0],
    [1, 1, 1, 1, 1, 1, 1, 0],
    [1, 1, 1, 0, 0, 1, 1, 0],
])

n_rows, n_cols = 3, 3
figsize = (15, 10)  # Adjust figure size to fit all subplots

fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize, clear=True, layout="constrained")
axes = axes.flatten()  # Flatten to easily index axes
k_values = range(0, 8)  # Values of k to iterate over
row_index = int(r[1:])  

for idx, k in enumerate(k_values):
    ax1 = axes[idx]
    # Check FSM matrix value to set background
    if FSM_with_NF[row_index][idx] == 1:
        ax1.set_facecolor('lightgrey')  # Set background for plots corresponding to 1

    # Plot Residuals on primary y-axis
    ax1.plot(all_r[k][ignore:], color='red', linestyle='--', label='Residuals')
    ax1.plot(2 * np.array(Thresholds[k][ignore:]), color='purple', linestyle='--', label='Adaptive Threshold')
    ax1.plot(-2 * np.array(Thresholds[k][ignore:]), color='purple', linestyle='--')

    # Set labels for ax1
    ax1.set_xlabel('x')
    ax1.set_ylabel('Residual/Aleatoric Uncertainty')  # Updated label
    ax1.legend(loc='upper left')
    ax1.set_title(f'{data_name[k]}')

# Plot the performance of the model on the training data
axes[8].plot(all_y_test[k][ignore:], label='Target', alpha=0.5, color='black')
axes[8].plot(all_mu_star[k][ignore:], color='black', label='$\mu^*$')
axes[8].plot(all_mu_star[k][ignore:] + 3 * all_sigma_star[k][ignore:], color='orange', linestyle='--', label='$\mu^* \pm 3*\sigma^*$')
axes[8].plot(all_mu_star[k][ignore:] - 3 * all_sigma_star[k][ignore:], color='orange', linestyle='--')
axes[8].legend()
axes[8].set_title('Performance of the model on the training data')

# Figure title
fig.suptitle(r, fontsize=16)
